In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.stats import norm, skewnorm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LinearRegression
from scipy.stats import rankdata
import warnings, os, time

warnings.filterwarnings("ignore")

MODEL_NAME = "TPL-3sig"
DATASET_NAME = "Yacht"
RESULTS_DIR = "results"
PLOTS_DIR = "plots"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

SPLIT_SEED = 58
SEEDS = [42, 58, 123, 256, 789]
N_RUNS = len(SEEDS)
DEVICE = torch.device("cpu")
torch.set_num_threads(1)

QUANTILES = [
    0.01, 0.025, 0.03, 0.05, 0.09, 0.10, 0.20, 0.30, 0.40, 0.50,
    0.60, 0.70, 0.80, 0.90, 0.91, 0.93, 0.95, 0.97, 0.975, 0.99,
]
OUTLIER_TYPES = ["gaussian", "multiply", "uniform", "skewed"]
CONTAMINATION_LEVELS = [0.02, 0.05, 0.10]
EP = 100; H = 100; LR = 0.01; BS = 32

plt.rcParams.update({
    "figure.figsize": (12, 5), "font.size": 11, "font.family": "sans-serif",
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False,
    "figure.dpi": 150, "savefig.dpi": 150, "savefig.bbox": "tight",
})
C_PRED = "#2563EB"; C_TRUE = "#DC2626"; C_CLEAN = "#3B82F6"
C_CONTAM = "#F59E0B"; C_MODEL = "#7C3AED"; C_BAR = "#0EA5E9"

print(f"Config: {MODEL_NAME} | {DATASET_NAME} | {N_RUNS} seeds | BS={BS} | Device: {DEVICE}")


Config: TPL-3sig | Yacht | 5 seeds | BS=32 | Device: cpu


In [2]:
# ── Data Loading: Yacht ──
try:
    from sklearn.datasets import fetch_openml
    data = fetch_openml(name="yacht_hydrodynamics", version=1, as_frame=True, parser="auto")
    df = data.frame
    X = df.iloc[:, :-1].values.astype(float)
    y = df.iloc[:, -1].values.astype(float)
except Exception:
    df = pd.read_csv("yacht.csv")
    X = df.iloc[:, :-1].values.astype(float)
    y = df.iloc[:, -1].values.astype(float)

scaler = StandardScaler()
X = scaler.fit_transform(X)
DIM = X.shape[1]
print(f"{DATASET_NAME}: {X.shape[0]} samples, {DIM} features")

Xtv, X_test, ytv, y_test = train_test_split(X, y, test_size=0.20, random_state=SPLIT_SEED)
X_train, X_val, y_train_clean, y_val = train_test_split(Xtv, ytv, test_size=0.20, random_state=SPLIT_SEED)
print(f"Split: Train={X_train.shape[0]} Val={X_val.shape[0]} Test={X_test.shape[0]}")

Yacht: 308 samples, 6 features
Split: Train=196 Val=50 Test=62


In [3]:
def inject_outliers(y, frac, method, seed=None):
    rng = np.random.RandomState(seed)
    y = y.copy()
    n = len(y)
    n_out = max(1, int(frac * n))
    idx = rng.choice(n, n_out, replace=False)
    mu, sig = y.mean(), y.std()
    if method == "gaussian":
        y[idx] = rng.normal(mu, np.sqrt(5) * sig, n_out)
    elif method == "multiply":
        y[idx] = y[idx] * rng.choice([-1, 1], n_out) * 10
    elif method == "uniform":
        y[idx] = rng.uniform(y.min() - 3 * sig, y.max() + 3 * sig, n_out)
    elif method == "skewed":
        y[idx] = skewnorm.rvs(a=10, loc=mu, scale=3 * sig, size=n_out, random_state=rng)
    return y

cont_sets = {}
for ot in OUTLIER_TYPES:
    for cl in CONTAMINATION_LEVELS:
        cont_sets[(ot, cl)] = inject_outliers(y_train_clean, cl, ot, seed=SPLIT_SEED)
print(f"{len(cont_sets)} contaminated training sets created")


12 contaminated training sets created


In [4]:
class MLP(nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, H), nn.ReLU(), nn.Linear(H, 1))
    def forward(self, x):
        return self.net(x).squeeze(-1)

def set_seed(s):
    np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def predict(model, X):
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(X, dtype=torch.float32).to(DEVICE)).cpu().numpy()

def train_model(X, y, loss_fn, epochs=EP):
    model = MLP(DIM).to(DEVICE)
    opt = optim.Adam(model.parameters(), lr=LR)
    Xt = torch.tensor(X, dtype=torch.float32).to(DEVICE)
    yt = torch.tensor(y, dtype=torch.float32).to(DEVICE)
    dl = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(Xt, yt), batch_size=BS, shuffle=True)
    model.train()
    for _ in range(epochs):
        for xb, yb in dl:
            opt.zero_grad(); loss_fn(model(xb), yb).backward(); opt.step()
    model.eval()
    return model

def train_mse(X, y, epochs=EP):
    return train_model(X, y, lambda p, t: nn.functional.mse_loss(p, t), epochs)

def pinball_loss(pred, target, tau):
    e = target - pred
    return torch.mean(torch.max(tau * e, (tau - 1) * e))

def tpl_loss(pred, target, tau, alpha):
    u = target - pred
    a = alpha
    return torch.mean(torch.where(
        (u >= 0) & (u < a * (1 - tau)), tau * u,
        torch.where(u >= a * (1 - tau),
            torch.tensor(tau * (1 - tau) * a, device=u.device, dtype=u.dtype),
            torch.where((u < 0) & (u >= -tau * a), (tau - 1) * u,
                torch.tensor(tau * (1 - tau) * a, device=u.device, dtype=u.dtype)))))

print("TPL-3sig defined")


TPL-3sig defined


In [5]:
def compute_shift_rmse(preds_clean, preds_contam, quantiles):
    results = {}
    for tau in quantiles:
        diff = preds_clean[tau] - preds_contam[tau]
        results[tau] = np.sqrt(np.mean(diff ** 2))
    return results

def compute_ece(y_test, preds, quantiles):
    results = {}
    for tau in quantiles:
        coverage = np.mean(y_test <= preds[tau])
        results[tau] = abs(coverage - tau)
    return results
print("Evaluation functions defined")


Evaluation functions defined


In [6]:
all_preds = {}; all_shift_rmse = {}; all_ece_clean = {}; all_ece_contam = {}
all_alphas = {}
conditions = ["clean"] + [(ot, cl) for ot in OUTLIER_TYPES for cl in CONTAMINATION_LEVELS]
total = N_RUNS * len(conditions) * len(QUANTILES)
print(f"Total models to train: {total}"); t0 = time.time()

for si, seed in enumerate(SEEDS):
    print(f"\n{'='*60}\nSeed {seed} ({si+1}/{N_RUNS})\n{'='*60}")
    all_preds[seed] = {}; all_shift_rmse[seed] = {}; all_ece_contam[seed] = {}
    all_alphas[seed] = {}
    for ci, cond in enumerate(conditions):
        cond_label = "clean" if cond == "clean" else f"{cond[0]}_{int(cond[1]*100)}pct"
        y_tr = y_train_clean if cond == "clean" else cont_sets[cond]
        set_seed(seed)
        mse_model = train_mse(X_train, y_tr)
        residuals = y_tr - predict(mse_model, X_train)
        alpha_fixed = 3.0 * np.std(residuals)
        all_alphas[seed][cond] = alpha_fixed
        preds = {}
        for tau in QUANTILES:
            set_seed(seed)
            model = train_model(X_train, y_tr, lambda p, y, t=tau, a=alpha_fixed: tpl_loss(p, y, t, a))
            preds[tau] = predict(model, X_test)
        all_preds[seed][cond] = preds
        if cond == "clean":
            all_ece_clean[seed] = compute_ece(y_test, preds, QUANTILES)
        else:
            all_shift_rmse[seed][cond] = compute_shift_rmse(all_preds[seed]["clean"], preds, QUANTILES)
            all_ece_contam[seed][cond] = compute_ece(y_test, preds, QUANTILES)
        done = si * len(conditions) * len(QUANTILES) + (ci+1) * len(QUANTILES)
        print(f"  [{done/total*100:5.1f}%] {cond_label:25s} α={alpha_fixed:.4f} | {(time.time()-t0)/60:.1f}min")
print(f"\nDone: {(time.time()-t0)/60:.1f} min")


Total models to train: 1300

Seed 42 (1/5)


  [  1.5%] clean                     α=2.0964 | 0.2min


  [  3.1%] gaussian_2pct             α=11.5332 | 0.4min


  [  4.6%] gaussian_5pct             α=15.7400 | 0.5min


  [  6.2%] gaussian_10pct            α=22.2718 | 0.7min


...

  [ 93.8%] uniform_5pct              α=27.5920 | 10.5min


  [ 95.4%] uniform_10pct             α=36.5233 | 10.7min


  [ 96.9%] skewed_2pct               α=17.8061 | 10.9min


  [ 98.5%] skewed_5pct               α=25.8336 | 11.1min


  [100.0%] skewed_10pct              α=31.0545 | 11.2min

Done: 11.2 min


In [7]:
from openpyxl import Workbook
wb = Workbook()
ws_summary = wb.active
ws_summary.title = "Summary"
contam_conditions = [(ot, cl) for ot in OUTLIER_TYPES for cl in CONTAMINATION_LEVELS]

for seed in SEEDS:
    ws = wb.create_sheet(title=f"ShiftRMSE_seed_{seed}")
    header = ["Condition"] + [str(t) for t in QUANTILES] + ["Avg"]
    ws.append(header)
    for cond in contam_conditions:
        label = f"{cond[0]}_{int(cond[1]*100)}pct"
        vals = all_shift_rmse[seed][cond]
        row = [label] + [round(vals[t], 6) for t in QUANTILES]
        row.append(round(np.mean([vals[t] for t in QUANTILES]), 6))
        ws.append(row)

for seed in SEEDS:
    ws = wb.create_sheet(title=f"ECE_clean_seed_{seed}")
    ws.append(["Metric"] + [str(t) for t in QUANTILES] + ["Avg"])
    vals = all_ece_clean[seed]
    row = ["ECE_clean"] + [round(vals[t], 6) for t in QUANTILES]
    row.append(round(np.mean([vals[t] for t in QUANTILES]), 6))
    ws.append(row)

for seed in SEEDS:
    ws = wb.create_sheet(title=f"ECE_contam_seed_{seed}")
    ws.append(["Condition"] + [str(t) for t in QUANTILES] + ["Avg"])
    for cond in contam_conditions:
        label = f"{cond[0]}_{int(cond[1]*100)}pct"
        vals = all_ece_contam[seed][cond]
        row = [label] + [round(vals[t], 6) for t in QUANTILES]
        row.append(round(np.mean([vals[t] for t in QUANTILES]), 6))
        ws.append(row)

ws_summary.append(["=== Shift-RMSE (mean across seeds) ==="])
ws_summary.append(["Condition"] + [str(t) for t in QUANTILES] + ["Avg"])
for cond in contam_conditions:
    label = f"{cond[0]}_{int(cond[1]*100)}pct"
    means = [np.mean([all_shift_rmse[s][cond][t] for s in SEEDS]) for t in QUANTILES]
    ws_summary.append([label] + [round(m, 6) for m in means] + [round(np.mean(means), 6)])

ws_summary.append([])
ws_summary.append(["=== ECE Clean (mean across seeds) ==="])
ws_summary.append(["Metric"] + [str(t) for t in QUANTILES] + ["Avg"])
means_ece = [np.mean([all_ece_clean[s][t] for s in SEEDS]) for t in QUANTILES]
ws_summary.append(["ECE_clean"] + [round(m, 6) for m in means_ece] + [round(np.mean(means_ece), 6)])

ws_alpha = wb.create_sheet(title="Alpha")
ws_alpha.append(["Seed", "Condition", "alpha_fixed"])
for seed in SEEDS:
    for cond in ["clean"] + [(ot, cl) for ot in OUTLIER_TYPES for cl in CONTAMINATION_LEVELS]:
        label = "clean" if cond == "clean" else f"{cond[0]}_{int(cond[1]*100)}pct"
        ws_alpha.append([f"seed_{seed}", label, round(all_alphas[seed].get(cond, 0), 6)])

xlsx_path = f"{RESULTS_DIR}/{DATASET_NAME}_{MODEL_NAME}_results.xlsx"
wb.save(xlsx_path)
print(f"Saved: {xlsx_path}")


Saved: results/Yacht_TPL-3sig_results.xlsx


In [8]:
x_ticks = list(range(len(QUANTILES)))
x_labels = [str(q) for q in QUANTILES]
contam_conditions = [(ot, cl) for ot in OUTLIER_TYPES for cl in CONTAMINATION_LEVELS]

for cond in contam_conditions:
    cond_label = f"{cond[0].capitalize()} {int(cond[1]*100)}%"
    fname_tag = f"{cond[0]}_{int(cond[1]*100)}pct"
    per_seed = np.array([[all_shift_rmse[s][cond][t] for t in QUANTILES] for s in SEEDS])
    means, stds = per_seed.mean(0), per_seed.std(0)

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.errorbar(x_ticks, means, yerr=stds, fmt="o-", color=C_BAR, lw=2, markersize=5, capsize=3)
    ax.set_xticks(x_ticks); ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=8)
    ax.set_xlabel("Quantile τ"); ax.set_ylabel("Shift-RMSE")
    ax.set_title(f"{DATASET_NAME} — Shift-RMSE: {MODEL_NAME} ({cond_label})")
    plt.tight_layout(); plt.savefig(f"{PLOTS_DIR}/shift_rmse_{fname_tag}_avg.png"); plt.close()

    for seed in SEEDS:
        fig, ax = plt.subplots(figsize=(14, 5))
        vals = [all_shift_rmse[seed][cond][t] for t in QUANTILES]
        ax.plot(x_ticks, vals, "o-", color=C_BAR, lw=2, markersize=5)
        ax.set_xticks(x_ticks); ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=8)
        ax.set_xlabel("Quantile τ"); ax.set_ylabel("Shift-RMSE")
        ax.set_title(f"{DATASET_NAME} — Shift-RMSE: {MODEL_NAME} ({cond_label}, seed={seed})")
        plt.tight_layout(); plt.savefig(f"{PLOTS_DIR}/shift_rmse_{fname_tag}_seed{seed}.png"); plt.close()
print("Shift-RMSE plots saved")


Shift-RMSE plots saved


In [9]:
x_ticks = list(range(len(QUANTILES)))
x_labels = [str(q) for q in QUANTILES]
all_ece_conditions = ["clean"] + [(ot, cl) for ot in OUTLIER_TYPES for cl in CONTAMINATION_LEVELS]

for cond in all_ece_conditions:
    if cond == "clean":
        cond_label, fname_tag = "Clean", "clean"
        per_seed = np.array([[all_ece_clean[s][t] for t in QUANTILES] for s in SEEDS])
    else:
        cond_label = f"{cond[0].capitalize()} {int(cond[1]*100)}%"
        fname_tag = f"{cond[0]}_{int(cond[1]*100)}pct"
        per_seed = np.array([[all_ece_contam[s][cond][t] for t in QUANTILES] for s in SEEDS])
    means, stds = per_seed.mean(0), per_seed.std(0)

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.errorbar(x_ticks, means, yerr=stds, fmt="o-", color="#10B981", lw=2, markersize=5, capsize=3)
    ax.set_xticks(x_ticks); ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=8)
    ax.set_xlabel("Quantile τ"); ax.set_ylabel("ECE")
    ax.set_title(f"{DATASET_NAME} — ECE: {MODEL_NAME} ({cond_label})")
    plt.tight_layout(); plt.savefig(f"{PLOTS_DIR}/ece_{fname_tag}_avg.png"); plt.close()

    for seed in SEEDS:
        fig, ax = plt.subplots(figsize=(14, 5))
        vals = all_ece_clean[seed] if cond == "clean" else all_ece_contam[seed][cond]
        ax.plot(x_ticks, [vals[t] for t in QUANTILES], "o-", color="#10B981", lw=2, markersize=5)
        ax.set_xticks(x_ticks); ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=8)
        ax.set_xlabel("Quantile τ"); ax.set_ylabel("ECE")
        ax.set_title(f"{DATASET_NAME} — ECE: {MODEL_NAME} ({cond_label}, seed={seed})")
        plt.tight_layout(); plt.savefig(f"{PLOTS_DIR}/ece_{fname_tag}_seed{seed}.png"); plt.close()
print("ECE plots saved")


ECE plots saved


In [10]:
print(f"{'='*60}")
print(f"{MODEL_NAME} on {DATASET_NAME} — Complete")
print(f"{'='*60}")
print(f"Results: {RESULTS_DIR}/"); print(f"Plots: {PLOTS_DIR}/")

print(f"\n--- Avg ECE (clean) ---")
avg_ece = np.mean([np.mean([all_ece_clean[s][t] for t in QUANTILES]) for s in SEEDS])
print(f"  Overall: {avg_ece:.4f}")

print(f"\n--- Avg Shift-RMSE (multiply 10%) ---")
for tau in [0.025, 0.05, 0.10, 0.50, 0.90, 0.95, 0.975]:
    vals = [all_shift_rmse[s][("multiply", 0.10)][tau] for s in SEEDS]
    print(f"  τ={tau:.3f}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")


TPL-3sig on Yacht — Complete
Results: results/
Plots: plots/

--- Avg ECE (clean) ---
  Overall: 0.3662

--- Avg Shift-RMSE (multiply 10%) ---
  τ=0.025: 9.9010 ± 2.8959
  τ=0.050: 8.4973 ± 0.9421
  τ=0.100: 9.7476 ± 0.8601
  τ=0.500: 10.5077 ± 0.4145
  τ=0.900: 13.0851 ± 2.1883
  τ=0.950: 17.8610 ± 3.5475
  τ=0.975: 26.3781 ± 0.7468
